# Model Training & Validation
**Champion-Challenger | KS / Gini / SHAP | Out-of-Time Holdout**

Two models: a Decision Tree as the interpretable champion (every decision path auditable by a regulator) and XGBoost as the challenger. Validation follows SR 11-7 — time-ordered split, no random sampling, KS and Gini as primary metrics.

Run `01_eda_feature_engineering.ipynb` first.

In [ ]:
import os
os.environ.setdefault('NUMBA_DISABLE_CUDA', '1')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve)
import xgboost as xgb
import shap

_HERE     = os.path.dirname(os.path.abspath('__file__'))
OUT_DIR   = os.path.join(_HERE, '..', 'outputs') + os.sep
MODEL_DIR = os.path.join(_HERE, '..', 'models') + os.sep
os.makedirs(MODEL_DIR, exist_ok=True)

## Time-Aware Train/Validation Split

The dataset is ordered by `TransactionDT`. Using the last 20% as the holdout means the model is validated on transactions it never saw during training — the same condition it faces in production. Random splits leak future data into training on time-series fraud data.

In [ ]:
X_train = pd.read_parquet(OUT_DIR + 'X_train.parquet')
y_train = pd.read_parquet(OUT_DIR + 'y_train.parquet')['isFraud']
X_test  = pd.read_parquet(OUT_DIR + 'X_test.parquet')

with open(OUT_DIR + 'features.json') as f:
    FEATURES = json.load(f)

split_idx = int(len(X_train) * 0.8)
X_tr, X_val = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
y_tr, y_val = y_train.iloc[:split_idx], y_train.iloc[split_idx:]

print(f"Train : {len(X_tr):,} rows  |  fraud rate: {y_tr.mean()*100:.2f}%")
print(f"Valid : {len(X_val):,} rows  |  fraud rate: {y_val.mean()*100:.2f}%")

## Validation Metrics

Bank model validation reports use KS and Gini, not accuracy or F1. KS measures the maximum separation between fraud and legitimate score distributions. Gini is 2×AUC−1. Both have well-established SR 11-7 pass thresholds (KS > 0.40, Gini > 0.60).

In [ ]:
def bfsi_metrics(y_true, y_prob, label='Model'):
    auc_roc = roc_auc_score(y_true, y_prob)
    auc_pr  = average_precision_score(y_true, y_prob)
    gini    = 2 * auc_roc - 1

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks = round(float(np.max(tpr - fpr)), 4)

    print(f"\n{label}")
    print("-" * 45)
    print(f"  KS Statistic  : {ks:.4f}  {'✓ (>0.40)' if ks > 0.4 else '⚠ weak'}")
    print(f"  Gini          : {gini:.4f}  {'✓ (>0.60)' if gini > 0.6 else '⚠ weak'}")
    print(f"  AUC-ROC       : {auc_roc:.4f}")
    print(f"  AUC-PR        : {auc_pr:.4f}")

    return {'ks': ks, 'gini': gini, 'auc_roc': auc_roc, 'auc_pr': auc_pr}

## Champion: Decision Tree

Depth-8 tree with balanced class weights. The point of the champion isn't to win on metrics — it's to give the business an auditable baseline. Every split can be traced, printed, and explained to a regulator. The challenger has to beat it convincingly before it earns deployment.

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42
)
dt.fit(X_tr, y_tr)
dt_prob   = dt.predict_proba(X_val)[:, 1]
dt_metrics = bfsi_metrics(y_val, dt_prob, 'Champion: Decision Tree')

joblib.dump(dt, MODEL_DIR + 'champion_decision_tree.pkl')
print("\nModel saved.")

## Challenger: XGBoost

`scale_pos_weight` is set to the negative/positive ratio (~27.6x) so the model doesn't just learn to predict the majority class. Early stopping on AUC-PR — the right optimisation target for imbalanced data, unlike AUC-ROC which can look good even when precision is poor.

In [ ]:
neg, pos  = (y_tr == 0).sum(), (y_tr == 1).sum()
scale_pw  = neg / pos
print(f"Class imbalance ratio: {scale_pw:.1f}x  →  scale_pos_weight={scale_pw:.1f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,
    eval_metric='aucpr',
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_prob    = xgb_model.predict_proba(X_val)[:, 1]
xgb_metrics = bfsi_metrics(y_val, xgb_prob, 'Challenger: XGBoost')

joblib.dump(xgb_model, MODEL_DIR + 'challenger_xgboost.pkl')
print("\nModel saved.")

## Champion vs Challenger — Validation Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Champion vs Challenger — Model Validation', fontsize=13)

for prob, label, color in [
    (dt_prob,  'Champion (Decision Tree)', '#534AB7'),
    (xgb_prob, 'Challenger (XGBoost)',     '#1D9E75')
]:
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    axes[0].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', linewidth=2, color=color)

axes[0].plot([0,1],[0,1],'--', color='gray', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(fontsize=9)

for prob, label, color in [
    (dt_prob,  'Champion (Decision Tree)', '#534AB7'),
    (xgb_prob, 'Challenger (XGBoost)',     '#1D9E75')
]:
    prec, rec, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    axes[1].plot(rec, prec, label=f'{label} (AP={ap:.3f})', linewidth=2, color=color)

baseline = y_val.mean()
axes[1].axhline(y=baseline, color='gray', linestyle='--', linewidth=1,
                label=f'Random ({baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR + '02_champion_challenger.png', dpi=150, bbox_inches='tight')
plt.show()

## KS Score Distributions

KS visualised as the separation between fraud and legitimate score histograms. A good model pushes fraud scores right and legitimate scores left — that gap is the KS statistic.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('KS Score Distributions (fraud vs legitimate)', fontsize=12)

for ax, prob, label in [
    (axes[0], dt_prob,  'Champion: Decision Tree'),
    (axes[1], xgb_prob, 'Challenger: XGBoost')
]:
    legit_scores = prob[y_val == 0]
    fraud_scores = prob[y_val == 1]
    ax.hist(legit_scores, bins=50, alpha=0.6, color='#1D9E75', label='Legitimate', density=True)
    ax.hist(fraud_scores, bins=50, alpha=0.6, color='#D85A30', label='Fraud', density=True)
    fpr, tpr, _ = roc_curve(y_val, prob)
    ks = float(np.max(tpr - fpr))
    ax.set_title(f'{label}  |  KS = {ks:.4f}', fontsize=10)
    ax.set_xlabel('Fraud probability score')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR + '03_ks_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Explainability

SHAP decomposes each prediction into per-feature contributions. This is what satisfies SR 11-7's conceptual soundness requirement and gives the compliance team something to cite when a flagged transaction is disputed.

Running on a 2,000-row sample — SHAP on the full validation set would take ~10 minutes and the feature ranking is stable by 2k.

In [ ]:
sample_idx  = np.random.choice(len(X_val), size=min(2000, len(X_val)), replace=False)
X_shap      = X_val.iloc[sample_idx]

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Feature Importance — XGBoost Challenger', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + '04_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + '05_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Outputs

In [ ]:
metrics_summary = {
    'champion_decision_tree': dt_metrics,
    'challenger_xgboost':     xgb_metrics
}
with open(OUT_DIR + 'metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

shap_df = pd.DataFrame({
    'feature':       X_shap.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("Top 15 features by SHAP importance:")
print("-" * 45)
for _, row in shap_df.head(15).iterrows():
    bar = '█' * int(row['mean_abs_shap'] * 200)
    print(f"  {row['feature']:<28} {row['mean_abs_shap']:.4f}  {bar}")

shap_df.to_csv(OUT_DIR + 'shap_importance.csv', index=False)

test_prob = xgb_model.predict_proba(X_test)[:, 1]
sub = pd.read_csv(os.path.join(_HERE, '..', 'data', 'ieee-fraud-detection', 'sample_submission.csv'))
sub['isFraud'] = test_prob
sub.to_csv(OUT_DIR + 'submission.csv', index=False)

print(f"\nMetrics     → {OUT_DIR}metrics.json")
print(f"SHAP values → {OUT_DIR}shap_importance.csv")
print(f"Submission  → {OUT_DIR}submission.csv")